# Ejecución general - Proyecto Timaran

Este notebook ejecuta el flujo completo de limpieza de datos:

1. Extrae la **tabla maestra de productos** desde el archivo `Informacion sin limpiar.xlsx`.
2. Limpia la tabla `Tabla Analisis de ventas.xlsx`.
3. Cruza las ventas con la tabla maestra.
4. Genera el archivo final listo para Power BI.

La lógica pesada está en archivos `.py`. Este notebook queda solo como panel de ejecución y validación.

## 1. Importar clases del proyecto

Como este notebook está pensado para estar dentro de la carpeta `Source`, los archivos `.py` deben estar en la misma carpeta.

In [1]:
from pathlib import Path
import pandas as pd

from extraccion_tabla_maestra import ExtractorTablaMaestra
from limpieza_ventas_oop import LimpiadorVentas

## 2. Definir rutas del proyecto

La estructura recomendada es:

```text
Proyecto Timaran/
├── Data/
│   ├── Informacion sin limpiar.xlsx
│   ├── Tabla Analisis de ventas.xlsx
│   └── Tabla_Maestra_Productos.xlsx
├── Resultado/
└── Source/
    ├── extraccion_tabla_maestra.py
    ├── limpieza_ventas_oop.py
    └── Ejecucion_General.ipynb
```

Como el notebook está en `Source`, usamos `../Data` para subir un nivel y entrar a la carpeta `Data`.

In [2]:
# Carpeta base del proyecto vista desde Source
DATA_DIR = Path("../Data")
RESULTADO_DIR = Path("../Resultado")

# Archivos de entrada
ARCHIVO_INFO_SIN_LIMPIAR = DATA_DIR / "Informacion sin limpiar.xlsx"
ARCHIVO_VENTAS = DATA_DIR / "Tabla Analisis de ventas.xlsx"

# Archivo intermedio y archivo final
ARCHIVO_TABLA_MAESTRA = DATA_DIR / "Tabla_Maestra_Productos.xlsx"

print("Archivo información sin limpiar:", ARCHIVO_INFO_SIN_LIMPIAR)
print("Archivo ventas:", ARCHIVO_VENTAS)
print("Archivo tabla maestra:", ARCHIVO_TABLA_MAESTRA)
print("Carpeta resultado:", RESULTADO_DIR)

Archivo información sin limpiar: ..\Data\Informacion sin limpiar.xlsx
Archivo ventas: ..\Data\Tabla Analisis de ventas.xlsx
Archivo tabla maestra: ..\Data\Tabla_Maestra_Productos.xlsx
Carpeta resultado: ..\Resultado


## 3. Validar que existan los archivos de entrada

Esta celda ayuda a detectar rápido errores de ruta como `FileNotFoundError`.

In [3]:
archivos_a_validar = [ARCHIVO_INFO_SIN_LIMPIAR, ARCHIVO_VENTAS]

for archivo in archivos_a_validar:
    if archivo.exists():
        print(f"OK: {archivo}")
    else:
        print(f"NO ENCONTRADO: {archivo}")

OK: ..\Data\Informacion sin limpiar.xlsx
OK: ..\Data\Tabla Analisis de ventas.xlsx


## 4. Extraer tabla maestra de productos

Aquí se crea un objeto `ExtractorTablaMaestra`.

Ese objeto conoce:

- dónde está el archivo original,
- dónde debe guardar la tabla maestra,
- qué hoja debe leer,
- y qué método debe ejecutar.

In [4]:
extractor = ExtractorTablaMaestra(
    ruta_archivo=ARCHIVO_INFO_SIN_LIMPIAR,
    ruta_salida=ARCHIVO_TABLA_MAESTRA,
    hoja="Análisis de ventas",
    fila_encabezado=3
)

ruta_maestra_generada = extractor.ejecutar()

print("Tabla maestra generada en:", ruta_maestra_generada)

Tabla maestra generada en: ..\Data\Tabla_Maestra_Productos.xlsx


## 5. Revisar resumen de la tabla maestra

Esto permite validar rápidamente si el catálogo de productos quedó bien extraído.

In [5]:
display(extractor.resumen())
display(extractor.df_maestro.head())

,METRICA,VALOR
0,Productos en tabla maestra,2397
1,Códigos únicos,2397
2,Clasificación I únicas,14
3,Clasificación II únicas,93
4,Clasificación III únicas,208
5,Clasificación IV únicas,48


,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
0,0001,ALCOHOL EXTRA NEUTRO.,ALCOHOL,ALCOHOL,ALCOHOL,ALCOHOL
1,0005,CAJA MADERA,CAJA,MADERA,NO APLICA,NO APLICA
2,0006,CAJA PARA CILINDRO 1OZ NEGRA,CAJA,CILINDRO,1 OZ,NO APLICA
3,0007,CAJA PARA CILINDRO 1OZ PLATA,CAJA,CILINDRO,1 OZ,NO APLICA
4,0008,CAJA PARA CILINDRO 1OZ DORADO,CAJA,CILINDRO,1 OZ,NO APLICA


## 6. Limpiar ventas usando la tabla maestra

Aquí se crea el objeto `LimpiadorVentas`.

Este objeto se encarga de:

- leer ventas,
- extraer códigos,
- convertir meses a formato largo,
- eliminar nulos, ceros y negativos,
- cruzar con tabla maestra,
- exportar el Excel final para Power BI.

In [6]:
limpiador = LimpiadorVentas(
    ruta_ventas=ARCHIVO_VENTAS,
    ruta_maestro=ruta_maestra_generada,
    carpeta_salida=RESULTADO_DIR,
    hoja_ventas="Análisis de ventas",
    hoja_maestro="Productos"
)

ruta_archivo_final = limpiador.ejecutar()

print("Archivo final generado en:", ruta_archivo_final)

Archivo final generado en: ..\Resultado\Ventas_Limpias_PowerBI.xlsx


## 7. Revisar resumen de la limpieza

Esta hoja de resumen te ayuda a saber cuántos registros quedaron y si hubo productos sin código o sin clasificación.

In [7]:
display(limpiador.df_resumen)

,METRICA,VALOR
0,Registros finales,26142.0
1,Cantidad total facturada,406136254.6
2,Productos sin código,2.0
3,Productos no clasificados,12.0
4,Periodos procesados,14.0
5,Productos clasificados únicos,2395.0


## 8. Vista previa del archivo final

Esta celda muestra las primeras filas del resultado que irá a Power BI.

In [8]:
display(limpiador.df_final.head(20))

,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV,PERIODO,CANTIDAD_FACTURADA
0,0001,ALCOHOL EXTRA NEUTRO.,ALCOHOL,ALCOHOL,ALCOHOL,ALCOHOL,ABRIL 2026,32190.0
1,2780,CAJA CILINDRICA BLANCA 100 ML X 120 UND,CAJA,CILINDRICA,100 ML,NO APLICA,ABRIL 2026,120.0
2,0008,CAJA PARA CILINDRO 1OZ DORADO,CAJA,CILINDRO,1 OZ,NO APLICA,ABRIL 2026,1100.0
3,0009,CAJA PARA CILINDRO 1OZ FUCSIA,CAJA,CILINDRO,1 OZ,NO APLICA,ABRIL 2026,1750.0
4,0006,CAJA PARA CILINDRO 1OZ NEGRA,CAJA,CILINDRO,1 OZ,NO APLICA,ABRIL 2026,1750.0
5,0007,CAJA PARA CILINDRO 1OZ PLATA,CAJA,CILINDRO,1 OZ,NO APLICA,ABRIL 2026,550.0
6,0010,CAJA PARA CILINDRO 2OZ DORADO,CAJA,CILINDRO,2 OZ,NO APLICA,ABRIL 2026,400.0
7,0011,CAJA PARA CILINDRO 2OZ FUCSIA,CAJA,CILINDRO,2 OZ,NO APLICA,ABRIL 2026,650.0
8,0012,CAJA PARA CILINDRO 2OZ NEGRA,CAJA,CILINDRO,2 OZ,NO APLICA,ABRIL 2026,800.0
9,0013,CAJA PARA CILINDRO 2OZ PLATA,CAJA,CILINDRO,2 OZ,NO APLICA,ABRIL 2026,100.0


## 9. Revisar productos con problemas

Estas tablas sirven para control de calidad:

- `productos_sin_codigo`: productos que no tenían código entre corchetes.
- `productos_no_clasificados`: productos que están en ventas, pero no aparecieron en la tabla maestra.

In [9]:
print("Productos sin código:", len(limpiador.productos_sin_codigo))
display(limpiador.productos_sin_codigo.head(20))

print("Productos no clasificados:", len(limpiador.productos_no_clasificados))
display(limpiador.productos_no_clasificados.head(20))

Productos sin código: 2


,PRODUCTO
0,FLETE INTERNACIONAL
1,Transporte


Productos no clasificados: 12


,CODIGO_PRODUCTO,PRODUCTO
0,0002,[0002] ALCOHOL EXTRA NEUTRO. X 1 LTR
1,0202,[0202] PREFORMA NO. 3 ENVASE PINK - JAMES - NE...
2,1147,[1147] MOOD VAINILLA DM
3,1553,[1553] Transporte
4,1685,[1685] ESENCIAS
5,1686,[1686] STICKER
6,1817,[1817] Arrendamiento Bodega
7,2245,[2245] Tramite
8,240101010003,[240101010003] ALCOHOL EXTRA NEUTRO. X GALON
9,2565,[2565] Banking FEE


## 10. Resultado

El archivo final queda en la carpeta `Resultado` con el nombre:

```text
Ventas_Limpias_PowerBI.xlsx
```

Ese archivo es el que puedes conectar a Power BI.